In [2]:
import os
import json
import uuid
from typing import Dict, Any, List, Tuple
from groq import Groq
from graphviz import Digraph

SYSTEM_PROMPT = (
    "You are an AI assistant specialized in creating detailed and informative mindmaps. "
    "When given a topic, generate a structured mindmap as a JSON object. "
    "Include descriptive titles and informative descriptions for each node. "
    "Aim for depth and breadth in the mindmap structure."
)

USER_TEMPLATE = (
    'Create a comprehensive and detailed mindmap for the topic: "{topic}". '
    "Include a main title, a root node, and at least three levels of child nodes where applicable. "
    "Provide informative descriptions for each node. "
    "Return ONLY valid JSON (no prose) following this schema:\n"
    "{\n"
    '  "title": "Main Title",\n'
    '  "description": "High-level overview",\n'
    '  "children": [\n'
    "    {\n"
    '      "title": "Subtopic 1",\n'
    '      "description": "Subtopic 1 info",\n'
    '      "children": [\n'
    "        { \"title\": \"Detail\", \"description\": \"...\", \"children\": [] }\n"
    "      ]\n"
    "    }\n"
    "  ]\n"
    "}\n"
)

def _parse_json_loose(text: str) -> Dict[str, Any]:
    # Try strict JSON first
    try:
        return json.loads(text)
    except Exception:
        pass
    # Try fenced code block ```json ... ```
    import re
    m = re.search(r"```json\s*([\s\S]*?)```", text, re.IGNORECASE)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    # Try first JSON object substring
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            pass
    raise ValueError("Failed to parse JSON from model output")

def generate_mindmap_json(topic: str, model: str = None) -> Dict[str, Any]:
    """
    Calls Groq to generate a hierarchical mindmap JSON for the given topic.
    Requires GROQ_API_KEY in environment.
    """
    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    model = model or os.getenv("GROQ_MODEL", "gemma-7b-it")

    user_prompt = USER_TEMPLATE.format(topic=topic)
    res = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    content = (res.choices[0].message.content or "").strip()
    data = _parse_json_loose(content)

    # Minimal normalization
    def _norm(node: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "title": str(node.get("title", "Untitled")).strip(),
            "description": str(node.get("description", "")).strip(),
            "children": [ _norm(c) for c in (node.get("children") or []) ],
        }
    return _norm(data)

def render_mindmap_image(mindmap: Dict[str, Any], out_path: str = "mindmap.png") -> str:
    """
    Renders a mindmap JSON structure to a PNG image using graphviz.
    Requires `graphviz` to be installed (both Python package and system binary `dot`).
    """
    dot = Digraph("mindmap", format="png")
    dot.attr(rankdir="TB", splines="spline", nodesep="0.4", ranksep="0.6")
    dot.attr("node", shape="box", style="rounded,filled", color="#4f46e5", fillcolor="#eef2ff", fontname="Helvetica", fontsize="10")
    dot.attr("edge", color="#64748b")

    def _label(title: str, desc: str, max_len: int = 80) -> str:
        # Multi-line label with soft wrap
        def wrap(txt: str, width: int) -> str:
            words, line, out = txt.split(), [], []
            for w in words:
                if sum(len(x) for x in line) + len(line) + len(w) <= width:
                    line.append(w)
                else:
                    out.append(" ".join(line))
                    line = [w]
            if line:
                out.append(" ".join(line))
            return out or [txt]
        desc_lines = wrap(desc, 50)[:6]
        desc_text = "\\n".join(desc_lines)
        title = title.strip() or "Untitled"
        if desc_text:
            return f"{title}\\n{desc_text}"
        return title

    def _add_nodes_edges(node: Dict[str, Any], parent_id: str = None) -> str:
        nid = str(uuid.uuid4())
        dot.node(nid, _label(node.get("title",""), node.get("description","")))
        for child in node.get("children", []):
            cid = _add_nodes_edges(child, nid)
            dot.edge(nid, cid)
        return nid

    _add_nodes_edges(mindmap)
    base = os.path.splitext(out_path)[0]
    rendered_path = dot.render(base, cleanup=True)
    # graphviz adds .png; ensure return is the png path
    if not rendered_path.endswith(".png"):
        rendered_path = f"{base}.png"
    return rendered_path

def generate_mindmap_image(topic: str, model: str = None, out_path: str = "mindmap.png") -> Tuple[Dict[str, Any], str]:
    """
    Convenience wrapper: generate mindmap JSON via Groq, then render to image.
    Returns (mindmap_json, image_path).
    """
    mindmap = generate_mindmap_json(topic, model=model)
    image_path = render_mindmap_image(mindmap, out_path=out_path)
    return mindmap, image_path

ModuleNotFoundError: No module named 'groq'

In [2]:
!pip install PyPDF2
!pip install ipykernal

ERROR: Could not find a version that satisfies the requirement ipykernal (from versions: none)
ERROR: No matching distribution found for ipykernal


In [3]:
!pip install langchain-groq faiss-cpu sentence-transformers

  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   -- ------------------------------------- 1.3/18.2 MB 15.8 MB/s eta 0:00:02
   -------- ------------------------------- 3.7/18.2 MB 8.3 MB/s eta 0:00:02
   ----------- ---------------------------- 5.2/18.2 MB 7.8 MB/s eta 0:00:02
   ---------------- ----------------------- 7.3/18.2 MB 8.5 MB/s eta 0:00:02
   ----------------- ---------------------- 7.9/18.2 MB 7.6 MB/s eta 0:00:02
   ------------------- -------------------- 8.9/18.2 MB 6.8 MB/s eta 0:00:02
   --------------------- ------------------ 9.7/18.2 MB 6.3 MB/s eta 0:00:02
   ---------------------- ----------------- 10.2/18.2 MB 6.1 MB/s eta 0:00:02
   ------------------------ --------------- 11.0/18.2 MB 5.7 MB/s eta 0:00:02
   ------------------------- -------------- 11.8/18.2 MB 5.4 MB/s eta 0:00:02
   --------------------------- ------------ 12.6/18.2 MB 5.2 MB/s eta 0:00:02
   

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

def chunk_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

corpus = chunk_text(text)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embeddings = embedding_model.encode(corpus, convert_to_numpy=True)

index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
index.add(np.array(corpus_embeddings))

def rag_search(query: str, top_k: int = 3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(np.array(query_embedding), k=top_k)
    results = [corpus[i] for i in indices[0]]
    return results


In [14]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000021A8AB5BF90> >

In [8]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0,api_key="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")

In [12]:
def generate_answer(query):
    retrieved_chunks = rag_search(query, top_k=3)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are a helpful and knowledgeable AI assistant for Atlantis Super Speciality Hospital.

You have two sources of information:
1. Context retrieved from hospital documents (below)
2. Your own medical and general knowledge

INSTRUCTIONS:
- First, look at the context. If it clearly answers the user's question, use it.
- If the context is incomplete or unrelated, use your own knowledge to fill in or answer completely.
- Always mention information from the hospital PDF first when relevant.
- Be clear, professional, and factual.

Context:
{context}

User Question:
{query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content


In [13]:
print(generate_answer("i want to know about force what it is"))


I couldn't find any information about "force" in the provided context. However, I can provide information on the topic based on my general knowledge.

Force can refer to several concepts depending on the context. Here are a few possible interpretations:

1. **Physics**: In physics, force is a push or pull that causes an object to change its motion or shape. It is a fundamental concept in mechanics and is often measured in units such as Newtons (N).
2. **Medical**: In medicine, force can refer to the amount of pressure or energy applied to a patient or a medical device. For example, a forceps is a medical instrument used to grasp or hold objects, and a force gauge is used to measure the force applied to a patient's skin or tissues.
3. **General**: In a general sense, force can refer to a strong or powerful influence that affects someone or something. For example, a person may be under the force of a strong emotion or a social force may influence a person's behavior.

If you could provid